# Spotify RAG-projekt


In [6]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_chroma import Chroma
from dotenv import load_dotenv
import pandas as pd
import os

### 2. Läs in datasetet med pandas

CSV-filen läses in med semikolon som separator och kolumnen `Unnamed: 0` tas bort.

In [13]:
df = pd.read_csv("data/dataset_spotify.csv", sep = ";", encoding="utf-8",low_memory=False)
print(df.head())

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [14]:
# Dropa Unnamed: 0 kolumn
df = df.drop(columns=["Unnamed: 0"])

print(df.head())
 
#spara nya csv filen
df.to_csv("data/cleaned_spotify.csv", sep=";", index=False, encoding="utf-8-sig")

                 track_id                 artists  \
0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   
3  Can't Help Falling In Love          71       201933     False   
4   

### 3. Ange sökväg till CSV-filen

Spara sökvägen till datasetet i variabeln `csv_path`.

In [4]:
csv_path = "data/cleaned_spotify.csv"

### 4. Ladda API-nyckel från `.env`

In [5]:
# Load your API key

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if api_key is None:
    print("API-nyckeln kunde inte hittas.")
else:
    print("API-nyckel laddad.")

API-nyckel laddad.


### 5. Skapa embedding-modellen som gör om text till vektorer.

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3930.29it/s]


### 6. Ladda CSV med rätt delimiter

In [7]:
loader = CSVLoader(file_path=csv_path, encoding="utf-8", csv_args={"delimiter": ";"})
documents = loader.load()

print(documents[0])

page_content='track_id: 5SuOikwiRyPMVoIQDJUgSV
artists: Gen Hoshino
album_name: Comedy
track_name: Comedy
popularity: 73
duration_ms: 230666
explicit: False
danceability: 0.676
energy: 0.461
key: 1
loudness: -6.746
mode: 0
speechiness: 0.143
acousticness: 0.0322
instrumentalness: 1.01e-06
liveness: 0.358
valence: 0.715
tempo: 87.917
time_signature: 4
track_genre: acoustic' metadata={'source': 'data/cleaned_spotify.csv', 'row': 0}


### 7. Splitta dokumenten och dela upp i mindre chunks innan embeddings skapas.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

all_splits = text_splitter.split_documents(documents)
subset = all_splits[:5000]


### 8. Skapa vektordatabas med Chroma

In [9]:
# Kontrollera om databasen finns, annars skapa

persist_directory = "./chroma_spotify_db"

if os.path.exists(persist_directory):
    print("Laddar befintlig ChromaDB...")
    vector_store = Chroma(
        collection_name="spotify",
        embedding_function=embeddings,
        persist_directory=persist_directory
    )

else:
    print("Skapar ny ChromaDB...")
    vector_store = Chroma.from_documents(
        documents=subset,
        embedding=embeddings,
        collection_name="spotify",
        persist_directory=persist_directory
    )

print("ChromaDB redo!")

Laddar befintlig ChromaDB...
ChromaDB redo!


### 9. Skapa en enkel sammanfattning av datasetet

In [10]:
summary_text = f"""
Dataset summary:
 
Number of rows: {len(df)}
Number of unique tracks: {df['track_id'].nunique()}
Duplicated track_id: {df.duplicated(subset=['track_id']).sum()}
Average popularity: {df['popularity'].mean():.2f}
Most common artist: {df['artists'].mode()[0]}
Average danceability: {df['danceability'].mean():.2f}
Missing values: {df.isnull().sum().sum()}
"""
 
print(summary_text)


Dataset summary:

Number of rows: 114000
Number of unique tracks: 89741
Duplicated track_id: 24259
Average popularity: 33.24
Most common artist: The Beatles
Average danceability: 0.57
Missing values: 3



### 10. Skapa retriever och prompt-template för RAG-systemet

En prompt-template som instruerar AI-modellen hur den ska svara på frågor.

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [12]:
template = """
You are a helpful assistant helping the user find different songs with the same artist

Instructions: 
Answer questions and use the dataset_spotify as reference
If you do not find the answer to the question, please answer that you don't know the answer.
unique albums, no duplicates

Context:
{context}

Question:
{question}
"""

prompt= PromptTemplate.from_template(template)

### 11. Testa similarity search

En fråga och det mest liknande dokumenten hämtas från vektordatabasen.

In [13]:
question = "What songs have high danceability"

# question = "Jason Mraz"

results = vector_store.similarity_search(question, k=20)

seen = set()

for res in results:

    content = res.page_content.strip()

    if content not in seen:
        seen.add(content)

        print(content)
        print("------")

track_id: 34p2uL1CvmaHn3uLfOKt8d
artists: Us The Duo
album_name: Public Record
track_name: One Last Dance
popularity: 51
duration_ms: 224847
explicit: False
danceability: 0.466
energy: 0.161
key: 7
loudness: -14.498
mode: 1
speechiness: 0.0492
acousticness: 0.979
instrumentalness: 0.000238
liveness: 0.231
valence: 0.171
tempo: 139.167
time_signature: 3
track_genre: acoustic
------
track_id: 4kbj5MwxO1bq9wjT5g9HaA
artists: WALK THE MOON
album_name: TALKING IS HARD
track_name: Shut Up and Dance
popularity: 83
duration_ms: 199080
explicit: False
danceability: 0.578
energy: 0.866
key: 1
loudness: -3.804
mode: 1
speechiness: 0.0619
acousticness: 0.00701
instrumentalness: 0.0
liveness: 0.257
valence: 0.619
tempo: 128.038
time_signature: 4
track_genre: alternative
------
track_id: 2rjtsklbFGd1qEwqt1rkE5
artists: Baby Tate
album_name: Sailor on the Moon - Hot New Rap
track_name: Dancing Queen
popularity: 0
duration_ms: 193076
explicit: True
danceability: 0.908
energy: 0.559
key: 11
loudness: -